# Lab 18. Probabilistic Forecasting and Uncertainty Quantification

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-18-probabilistic-forecasting-uncertainty-lab.ipynb)

This lab is designed for independent study. You should be able to read the background, run each code cell, and answer the interpretation questions without needing additional setup.

## Learning goals

By the end of this lab, you will be able to:

1. Explain why a point forecast is not enough for many decisions.
2. Construct prediction intervals using residuals, Gaussian approximations, bootstrap residuals, and split conformal prediction.
3. Evaluate interval forecasts using coverage and average width.
4. Train quantile regression models for lower and upper predictive quantiles.
5. Use pinball loss to evaluate quantile forecasts.
6. Convert a predictive distribution into a decision probability, such as the probability that demand exceeds a threshold.

## Why probabilistic forecasting?

A point forecast gives one number, such as

$$
\hat y_{t+1} = 42.7.
$$

A probabilistic forecast tries to describe uncertainty around future values. For example, it may report

$$
P(39 \le Y_{t+1} \le 47 \mid \text{available information}) \approx 0.90.
$$

The distinction is important. If the forecast is used for staffing, inventory, medical monitoring, finance, or energy planning, the decision-maker usually needs to know not only the expected value but also the uncertainty and tail risk.

Throughout this lab, let the information available at time $t$ be denoted by $F_t$. A point forecast estimates something like

$$
E(Y_{t+1} \mid F_t),
$$

while a probabilistic forecast describes the conditional distribution of $Y_{t+1}$ given $F_t$.

> **Colab note.** This notebook avoids the LaTeX command that sometimes displays poorly in Colab.

## 1. Setup

We use only common Python packages available in Google Colab: `numpy`, `pandas`, `matplotlib`, `scipy`, and `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(7339)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

## 2. Simulate a time series with predictable structure and changing uncertainty

We will create a synthetic series with four ingredients:

1. A slowly increasing trend.
2. A daily seasonal pattern.
3. Occasional event effects.
4. Autocorrelated noise with a variance that changes slowly over time.

This is useful for studying uncertainty because the model should learn the mean pattern, but the remaining error is still random.

The data-generating idea is

$$
Y_t = m_t + E_t,
$$

where $m_t$ is the predictable mean pattern and $E_t$ is a dependent noise process.

In [ ]:
n = 720
period = 24
t = np.arange(n)

# Known calendar/event variables.
event = ((t % 90) >= 0) & ((t % 90) <= 2)
event = event.astype(float)

trend = 0.008 * t
daily = 2.0 * np.sin(2 * np.pi * t / period) + 0.9 * np.cos(2 * np.pi * t / period)
weekly = 0.7 * np.sin(2 * np.pi * t / (7 * period))
mean_signal = 20 + trend + daily + weekly + 1.8 * event

# Changing noise scale: uncertainty is larger later in the series.
sigma_t = 0.55 + 0.0009 * t + 0.25 * (t > 520)
errors = np.zeros(n)
for i in range(1, n):
    errors[i] = 0.55 * errors[i - 1] + sigma_t[i] * rng.normal()

y = mean_signal + errors

data = pd.DataFrame({
    "t": t,
    "y": y,
    "mean_signal": mean_signal,
    "sigma_t": sigma_t,
    "event": event
})

data.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(data["t"], data["y"], label="observed series", linewidth=1.2)
ax.plot(data["t"], data["mean_signal"], label="hidden mean signal", linewidth=2)
ax.set_title("Synthetic time series: signal plus changing uncertainty")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(data["t"], data["sigma_t"], label="true noise scale")
ax.set_title("The data become more uncertain over time")
ax.set_xlabel("time")
ax.set_ylabel("noise scale")
ax.legend()
plt.show()

### Interpretation checkpoint

Look at the first plot and the noise-scale plot.

1. Is the mean pattern constant over time?
2. Does uncertainty appear constant over time?
3. Why might a single global interval width be too narrow or too wide in some parts of the series?

## 3. Build forecasting features without leakage

We forecast one step ahead. At time $t$, the forecast for $Y_t$ may use earlier observed values such as $Y_{t-1}, Y_{t-2}, \ldots$ and known calendar variables at time $t$.

A common supervised learning design matrix has rows of the form

$$
X_t = (Y_{t-1}, Y_{t-2}, \ldots, \text{calendar features at time } t),
$$

with response $Y_t$.

A feature is valid only if it would be known at the time the forecast is made.

In [ ]:
def make_features(df, max_lag=24, period=24):
    out = df.copy()
    for lag in range(1, max_lag + 1):
        out[f"lag_{lag}"] = out["y"].shift(lag)

    # Known time features.
    out["time_scaled"] = out["t"] / out["t"].max()
    out["sin_daily"] = np.sin(2 * np.pi * out["t"] / period)
    out["cos_daily"] = np.cos(2 * np.pi * out["t"] / period)
    out["sin_weekly"] = np.sin(2 * np.pi * out["t"] / (7 * period))
    out["cos_weekly"] = np.cos(2 * np.pi * out["t"] / (7 * period))

    out = out.dropna().reset_index(drop=True)
    feature_cols = [c for c in out.columns if c.startswith("lag_")]
    feature_cols += ["time_scaled", "sin_daily", "cos_daily", "sin_weekly", "cos_weekly", "event"]
    return out, feature_cols

model_data, feature_cols = make_features(data, max_lag=24, period=24)

print("Number of supervised rows:", len(model_data))
print("Number of features:", len(feature_cols))
model_data[["t", "y"] + feature_cols[:5]].head()

## 4. Chronological train, calibration, and test split

For probabilistic forecasting, we often use three time-ordered parts:

1. **Training set:** fit the point forecasting model.
2. **Calibration set:** estimate uncertainty from recent forecast errors.
3. **Test set:** evaluate future performance.

The calibration set must come after the training set and before the test set. Shuffling would leak future information into the past.

In [ ]:
train_end = 440
cal_end = 580

train = model_data[model_data["t"] < train_end].copy()
cal = model_data[(model_data["t"] >= train_end) & (model_data["t"] < cal_end)].copy()
test = model_data[model_data["t"] >= cal_end].copy()

X_train, y_train = train[feature_cols], train["y"]
X_cal, y_cal = cal[feature_cols], cal["y"]
X_test, y_test = test[feature_cols], test["y"]

print(len(train), len(cal), len(test))
print("train time range:", int(train.t.min()), "to", int(train.t.max()))
print("calibration time range:", int(cal.t.min()), "to", int(cal.t.max()))
print("test time range:", int(test.t.min()), "to", int(test.t.max()))

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["t"], train["y"], label="train")
ax.plot(cal["t"], cal["y"], label="calibration")
ax.plot(test["t"], test["y"], label="test")
ax.set_title("Chronological split")
ax.set_xlabel("time")
ax.set_ylabel("y")
ax.legend()
plt.show()

## 5. Fit a point forecasting model

We first fit a ridge regression model. This gives a point forecast

$$
\hat y_t = f(X_t).
$$

Then we will add uncertainty around this point forecast.

In [ ]:
point_model = Ridge(alpha=3.0)
point_model.fit(X_train, y_train)

pred_train = point_model.predict(X_train)
pred_cal = point_model.predict(X_cal)
pred_test = point_model.predict(X_test)

rmse_test = mean_squared_error(y_test, pred_test) ** 0.5
mae_test = mean_absolute_error(y_test, pred_test)
print(f"Test MAE:  {mae_test:.3f}")
print(f"Test RMSE: {rmse_test:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test["t"], y_test.values, label="observed", linewidth=1.5)
ax.plot(test["t"], pred_test, label="point forecast", linewidth=2)
ax.set_title("Point forecasts on the test period")
ax.set_xlabel("time")
ax.set_ylabel("y")
ax.legend()
plt.show()

### Interpretation checkpoint

A good point forecast can still be insufficient.

1. Where does the point forecast miss the observations by a large amount?
2. Would the forecast be more useful if it came with a 90% prediction interval?
3. Why should interval quality be evaluated on the test period rather than on the training period?

## 6. Functions for interval evaluation

For a nominal 90% prediction interval, we want approximately 90% of future observations to fall inside the interval. Two basic interval diagnostics are:

- **Coverage:** fraction of observations inside the interval.
- **Average width:** average upper bound minus lower bound.

A very wide interval may have high coverage but be uninformative. A very narrow interval may look sharp but under-cover.

In [ ]:
def interval_summary(y_true, lower, upper, name="interval"):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    inside = (y_true >= lower) & (y_true <= upper)
    coverage = inside.mean()
    width = np.mean(upper - lower)
    return pd.Series({
        "method": name,
        "coverage": coverage,
        "average_width": width,
        "misses": int((~inside).sum())
    })

def plot_interval(t_values, y_true, pred, lower, upper, title):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t_values, y_true, label="observed", linewidth=1.4)
    ax.plot(t_values, pred, label="point forecast", linewidth=2)
    ax.fill_between(t_values, lower, upper, alpha=0.25, label="prediction interval")
    ax.set_title(title)
    ax.set_xlabel("time")
    ax.set_ylabel("y")
    ax.legend()
    plt.show()

## 7. Gaussian residual interval

A simple approach assumes that forecast errors are approximately Gaussian:

$$
Y_t - \hat y_t \approx N(0, \sigma^2).
$$

For a 90% interval, use

$$
\hat y_t \pm z_{0.95}\hat\sigma,
$$

where $z_{0.95}$ is the 95th percentile of a standard normal distribution. We estimate $\hat\sigma$ using calibration residuals, not test residuals.

In [ ]:
alpha = 0.10
z = norm.ppf(1 - alpha / 2)

cal_resid = y_cal.values - pred_cal
sigma_hat = cal_resid.std(ddof=1)

normal_lower = pred_test - z * sigma_hat
normal_upper = pred_test + z * sigma_hat

normal_summary = interval_summary(y_test, normal_lower, normal_upper, "Gaussian residual")
normal_summary

In [ ]:
plot_interval(
    test["t"].values,
    y_test.values,
    pred_test,
    normal_lower,
    normal_upper,
    "Gaussian residual 90% prediction interval"
)

### Interpretation checkpoint

1. Does the Gaussian interval appear to have constant width?
2. Does constant width make sense for this data-generating process?
3. What might go wrong if the forecast errors are skewed, heavy-tailed, or heteroskedastic?

## 8. Residual bootstrap interval

The Gaussian interval assumes a symmetric normal error distribution. A residual bootstrap interval uses empirical calibration residuals instead:

$$
\hat y_t + q_{\alpha/2}(R), \quad \hat y_t + q_{1-\alpha/2}(R),
$$

where $R$ is the set of calibration residuals.

This method is still simple, but it can reflect asymmetry in residuals.

In [ ]:
q_low, q_high = np.quantile(cal_resid, [alpha / 2, 1 - alpha / 2])
boot_lower = pred_test + q_low
boot_upper = pred_test + q_high

boot_summary = interval_summary(y_test, boot_lower, boot_upper, "Residual bootstrap")
boot_summary

In [ ]:
plot_interval(
    test["t"].values,
    y_test.values,
    pred_test,
    boot_lower,
    boot_upper,
    "Residual bootstrap 90% prediction interval"
)

## 9. Split conformal prediction

Split conformal prediction uses calibration errors to form an interval with finite-sample coverage under exchangeability assumptions.

For symmetric absolute residual intervals, define calibration scores

$$
S_i = |Y_i - \hat Y_i|.
$$

Let $q$ be a high quantile of these scores. Then the conformal interval is

$$
[\hat y_t - q, \hat y_t + q].
$$

For time series, exchangeability is not automatic because observations are dependent and the distribution can drift. Still, split conformal prediction is a useful baseline, especially when the calibration period is recent and representative of the test period.

In [ ]:
def conformal_quantile(scores, alpha=0.10):
    scores = np.sort(np.asarray(scores))
    m = len(scores)
    # Finite-sample split conformal quantile index.
    k = int(np.ceil((m + 1) * (1 - alpha)))
    k = min(max(k, 1), m)
    return scores[k - 1]

abs_scores = np.abs(cal_resid)
q_conf = conformal_quantile(abs_scores, alpha=alpha)

conf_lower = pred_test - q_conf
conf_upper = pred_test + q_conf

conf_summary = interval_summary(y_test, conf_lower, conf_upper, "Split conformal")
conf_summary

In [ ]:
plot_interval(
    test["t"].values,
    y_test.values,
    pred_test,
    conf_lower,
    conf_upper,
    "Split conformal 90% prediction interval"
)

## 10. Rolling conformal intervals

The previous conformal interval used one calibration window and gave every test point the same interval width. A more adaptive approach updates the residual window as new observations become available.

At each test time:

1. Use recent residual scores to choose $q$.
2. Construct the interval for the next test point.
3. After observing the true value, add the new absolute residual to the calibration pool.

This is not a magic solution, but it is often more realistic for time series with changing uncertainty.

In [ ]:
def rolling_conformal_intervals(y_true, pred, initial_scores, alpha=0.10, window=120):
    scores = list(np.asarray(initial_scores))
    lower = []
    upper = []
    q_values = []
    for yt, phat in zip(y_true, pred):
        recent_scores = np.asarray(scores[-window:])
        q = conformal_quantile(recent_scores, alpha=alpha)
        lower.append(phat - q)
        upper.append(phat + q)
        q_values.append(q)
        scores.append(abs(yt - phat))
    return np.array(lower), np.array(upper), np.array(q_values)

roll_lower, roll_upper, roll_q = rolling_conformal_intervals(
    y_test.values, pred_test, abs_scores, alpha=alpha, window=100
)

roll_summary = interval_summary(y_test, roll_lower, roll_upper, "Rolling conformal")
roll_summary

In [ ]:
plot_interval(
    test["t"].values,
    y_test.values,
    pred_test,
    roll_lower,
    roll_upper,
    "Rolling conformal 90% prediction interval"
)

fig, ax = plt.subplots()
ax.plot(test["t"].values, roll_q)
ax.set_title("Rolling conformal half-width over the test period")
ax.set_xlabel("time")
ax.set_ylabel("q")
plt.show()

## 11. Compare interval methods

The ideal method has coverage near the nominal level and small width. In practice, there is a tradeoff between reliability and sharpness.

In [ ]:
summaries = pd.DataFrame([
    normal_summary,
    boot_summary,
    conf_summary,
    roll_summary
])

summaries["coverage"] = summaries["coverage"].round(3)
summaries["average_width"] = summaries["average_width"].round(3)
summaries

### Interpretation checkpoint

1. Which method has coverage closest to 0.90?
2. Which method has the narrowest average width?
3. Which method would you prefer if missing a high value is very costly?
4. Which method would you prefer if false alarms are very costly?

## 12. Quantile regression

Instead of fitting a mean forecast and adding residual intervals, quantile regression directly estimates conditional quantiles.

For a lower quantile $q_{0.05}(X_t)$ and upper quantile $q_{0.95}(X_t)$, a 90% interval is

$$
[q_{0.05}(X_t), q_{0.95}(X_t)].
$$

We will use gradient boosting quantile regression from `scikit-learn`.

In [ ]:
# Fit quantile gradient boosting models.
# Small settings keep this Colab-friendly.
q05_model = GradientBoostingRegressor(
    loss="quantile", alpha=0.05, n_estimators=120, max_depth=2, learning_rate=0.05, random_state=1
)
q50_model = GradientBoostingRegressor(
    loss="quantile", alpha=0.50, n_estimators=120, max_depth=2, learning_rate=0.05, random_state=1
)
q95_model = GradientBoostingRegressor(
    loss="quantile", alpha=0.95, n_estimators=120, max_depth=2, learning_rate=0.05, random_state=1
)

q05_model.fit(X_train, y_train)
q50_model.fit(X_train, y_train)
q95_model.fit(X_train, y_train)

q05 = q05_model.predict(X_test)
q50 = q50_model.predict(X_test)
q95 = q95_model.predict(X_test)

# Ensure lower <= upper in case of small quantile crossing.
qr_lower = np.minimum(q05, q95)
qr_upper = np.maximum(q05, q95)
qr_pred = q50

qr_summary = interval_summary(y_test, qr_lower, qr_upper, "Quantile boosting")
qr_summary

In [ ]:
plot_interval(
    test["t"].values,
    y_test.values,
    qr_pred,
    qr_lower,
    qr_upper,
    "Gradient boosting quantile 90% prediction interval"
)

### Interpretation checkpoint

1. Does the quantile interval width vary over time?
2. Does the quantile model appear to adapt to larger uncertainty later in the series?
3. Why might direct quantile models perform poorly when the training set is small?

## 13. Pinball loss for quantile forecasts

The pinball loss evaluates a quantile forecast. For quantile level $\tau$, prediction $q$, and observation $y$, the loss is

$$
L_\tau(y,q) =
\begin{cases}
\tau(y-q), & y \ge q, \\
(1-\tau)(q-y), & y < q.
\end{cases}
$$

Lower pinball loss is better.

In [ ]:
def pinball_loss(y_true, q_pred, tau):
    y_true = np.asarray(y_true)
    q_pred = np.asarray(q_pred)
    diff = y_true - q_pred
    return np.mean(np.maximum(tau * diff, (tau - 1) * diff))

pinball_table = pd.DataFrame({
    "quantile": [0.05, 0.50, 0.95],
    "pinball_loss": [
        pinball_loss(y_test, q05, 0.05),
        pinball_loss(y_test, q50, 0.50),
        pinball_loss(y_test, q95, 0.95),
    ]
})

pinball_table

## 14. PIT diagnostic for Gaussian predictive distributions

If we claim that

$$
Y_t \mid F_{t-1} \sim N(\hat y_t, \hat\sigma^2),
$$

then the probability integral transform values

$$
U_t = \Phi\left(\frac{Y_t - \hat y_t}{\hat\sigma}\right)
$$

should look roughly uniform on $[0,1]$ when the predictive distribution is calibrated.

A U-shaped PIT histogram suggests under-dispersion. A hump-shaped PIT histogram suggests over-dispersion.

In [ ]:
pit = norm.cdf((y_test.values - pred_test) / sigma_hat)

fig, ax = plt.subplots()
ax.hist(pit, bins=10, edgecolor="black")
ax.axhline(len(pit) / 10, linestyle="--", label="rough uniform reference")
ax.set_title("PIT histogram for Gaussian residual forecast")
ax.set_xlabel("PIT value")
ax.set_ylabel("count")
ax.legend()
plt.show()

print("Mean PIT:", pit.mean().round(3))
print("Min/Max PIT:", pit.min().round(3), pit.max().round(3))

## 15. From forecasts to decisions: threshold risk

Suppose a manager wants to know whether the next value will exceed a high threshold. A predictive distribution can estimate

$$
P(Y_t > c \mid F_{t-1}).
$$

Under the Gaussian residual forecast,

$$
P(Y_t > c \mid F_{t-1}) = 1 - \Phi\left(\frac{c - \hat y_t}{\hat\sigma}\right).
$$

We will flag high-risk times when this probability exceeds 0.40.

In [ ]:
threshold = np.quantile(train["y"], 0.90)
risk_prob = 1 - norm.cdf((threshold - pred_test) / sigma_hat)
alert = risk_prob > 0.40
actual_exceed = y_test.values > threshold

risk_table = pd.DataFrame({
    "t": test["t"].values,
    "observed": y_test.values,
    "forecast": pred_test,
    "P_exceed": risk_prob,
    "alert": alert,
    "actual_exceed": actual_exceed
})

pd.crosstab(risk_table["alert"], risk_table["actual_exceed"], rownames=["alert"], colnames=["actual_exceed"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test["t"].values, risk_prob, label="estimated probability")
ax.axhline(0.40, linestyle="--", label="alert threshold")
ax.set_title(f"Estimated probability that Y exceeds {threshold:.2f}")
ax.set_xlabel("time")
ax.set_ylabel("probability")
ax.legend()
plt.show()

risk_table.head()

### Interpretation checkpoint

1. What does a false positive mean in this threshold decision problem?
2. What does a false negative mean?
3. Which error type is more costly in an inventory problem? In a medical monitoring problem?
4. How would you choose the alert probability threshold in a real project?

## 16. Mini-project: uncertainty-aware forecasting report

Choose one of the following tasks.

### Option A. Change the data-generating process

Modify the synthetic data so that uncertainty increases more sharply after time 500. Then rerun the interval methods.

Report:

1. Which method adapts best?
2. Which method under-covers?
3. Which method gives the best balance between coverage and width?

### Option B. Compare point models

Replace ridge regression with random forest or gradient boosting point forecasts. Then rerun the uncertainty methods.

Report:

1. Did better point forecasts also improve interval quality?
2. Did the calibration residual distribution change?
3. Did the conformal interval become narrower?

### Option C. Decision-focused evaluation

Choose a threshold $c$ and a probability alert rule. Evaluate false positives and false negatives.

Report:

1. What is the practical meaning of the threshold?
2. What is the practical meaning of an alert?
3. What is the cost of false positives and false negatives?

## 17. AI-assisted study prompts

Use these prompts after you have completed the lab. Do not use them as a substitute for running the code and interpreting the plots yourself.

### Prompt 1: Explain the difference between intervals

> I built Gaussian residual, residual bootstrap, split conformal, rolling conformal, and quantile regression intervals for a time series forecast. Explain the assumptions of each method and when each method may fail.

### Prompt 2: Diagnose under-coverage

> My 90% prediction interval only covers 75% of test observations. Give a checklist of possible causes specific to time series forecasting, including distribution drift, heteroskedasticity, leakage, and poor point forecasts.

### Prompt 3: Decision interpretation

> I estimate P(Y exceeds a threshold) for each future time point. Help me write a careful paragraph explaining how forecast uncertainty affects decision-making and why a point forecast alone is not enough.

## 18. Lab checklist

Before leaving this lab, make sure you can do the following:

- [ ] Explain why probabilistic forecasting is different from point forecasting.
- [ ] Build a chronological train/calibration/test split.
- [ ] Construct Gaussian residual prediction intervals.
- [ ] Construct residual bootstrap prediction intervals.
- [ ] Construct split conformal prediction intervals.
- [ ] Explain why time-series conformal prediction needs care when the distribution changes.
- [ ] Train a quantile regression model.
- [ ] Compute coverage and average interval width.
- [ ] Compute pinball loss for quantile forecasts.
- [ ] Convert a predictive distribution into a threshold-exceedance probability.